# NB08 — Climate Comparison & Wildfire Trend Analysis

**Objective:** Answer key questions for a jury-ready demo:

1. **Is the forecasted weather hotter than last year?** → compare next 30 days vs same period last year
2. **Is it hotter than the decade average?** → compare vs 2015–2024 same-month historical
3. **Is there a climate shift signal?** → long-term trends in temperature, extreme heat days, dry days
4. **Will wildfire risk increase?** → compare predicted risk vs last year and historical averages
5. **Which cities are highest risk and why?** → ranked summary with drivers

All conclusions are phrased cautiously ("signal", "indication") — not absolute claims.

**Input:** `outputs/wildfire_risk_30d.parquet`, `data/processed/engineered_daily.parquet`  
**Output:** `reports/figures/climate_*.png`, `reports/metrics/climate_summary.csv`

In [ ]:
# ─── Cell 1: Imports ─────────────────────────────────────────────────────
import os, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from src.config import (ROOT, PROCESSED, OUTPUTS, CITIES,
                         ENG_DAILY, MASTER_DAILY, RISK_30D)

FIGURES = ROOT / "reports" / "figures"
METRICS = ROOT / "reports" / "metrics"
for p in (FIGURES, METRICS):
    p.mkdir(parents=True, exist_ok=True)

print(f"Root: {ROOT}")

In [ ]:
# ─── §1: Load data ───────────────────────────────────────────────────────

# Historical daily data
hist_path = ENG_DAILY if ENG_DAILY.exists() else MASTER_DAILY
hist = pd.read_parquet(hist_path)
hist["Date"] = pd.to_datetime(hist["Date"])
hist = hist.sort_values(["City", "Date"])
print(f"Historical: {hist.shape} ({hist['Date'].min().date()} → {hist['Date'].max().date()})")

# Forecast risk data
risk = pd.read_parquet(RISK_30D)
risk["Date"] = pd.to_datetime(risk["Date"])
print(f"Risk forecast: {risk.shape} ({risk['Date'].min().date()} → {risk['Date'].max().date()})")

# Date ranges for comparison
fc_start = risk["Date"].min()
fc_end   = risk["Date"].max()
fc_month_start = fc_start.month
fc_month_end   = fc_end.month

# Last year same period
ly_start = fc_start - pd.DateOffset(years=1)
ly_end   = fc_end   - pd.DateOffset(years=1)

# Decade (2015-2024)
DECADE_START = 2015
DECADE_END   = 2024

print(f"\nForecast period:   {fc_start.date()} → {fc_end.date()}")
print(f"Last year period:  {ly_start.date()} → {ly_end.date()}")
print(f"Decade reference:  {DECADE_START}–{DECADE_END}, months {fc_month_start}–{fc_month_end}")

## §2 — Is the Forecast Hotter Than Last Year?

We compare the mean forecasted temperature for each city against the actual temperature from the same date range one year ago.

In [ ]:
# ─── §2: Forecast vs Last Year ───────────────────────────────────────────

# Get forecast temperature per city
temp_col = "Temperature_C_mean"
fc_temp = risk.groupby("City")[temp_col].mean() if temp_col in risk.columns else pd.Series(dtype=float)

# Get last year same period
ly_data = hist[(hist["Date"] >= ly_start) & (hist["Date"] <= ly_end)]
ly_temp = ly_data.groupby("City")[temp_col].mean() if temp_col in ly_data.columns else pd.Series(dtype=float)

# Compare
if not fc_temp.empty and not ly_temp.empty:
    comp_ly = pd.DataFrame({
        "Forecast_Temp": fc_temp,
        "LastYear_Temp": ly_temp,
    }).dropna()
    comp_ly["Difference"] = comp_ly["Forecast_Temp"] - comp_ly["LastYear_Temp"]
    comp_ly["Hotter"] = comp_ly["Difference"] > 0
    comp_ly = comp_ly.sort_values("Difference", ascending=False)

    print("Temperature: Forecast vs Last Year (same period)")
    print("=" * 65)
    for city, row in comp_ly.iterrows():
        arrow = "↑ HOTTER" if row["Hotter"] else "↓ cooler"
        print(f"  {city:15s}  Forecast={row['Forecast_Temp']:5.1f}°C  "
              f"LastYear={row['LastYear_Temp']:5.1f}°C  "
              f"Diff={row['Difference']:+.1f}°C  {arrow}")

    n_hotter = comp_ly["Hotter"].sum()
    print(f"\n  {n_hotter}/{len(comp_ly)} cities are forecast to be HOTTER than last year")
    avg_diff = comp_ly["Difference"].mean()
    print(f"  Average temperature difference: {avg_diff:+.2f}°C")

    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    x = range(len(comp_ly))
    colors = ["#e74c3c" if d > 0 else "#3498db" for d in comp_ly["Difference"]]
    ax.bar(x, comp_ly["Difference"], color=colors, alpha=0.8, edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels(comp_ly.index, rotation=45, ha="right")
    ax.set_ylabel("Temperature Difference (°C)")
    ax.set_title("Forecast vs Last Year Temperature — By City", fontsize=14)
    ax.axhline(0, color="black", linewidth=0.8)
    for i, v in enumerate(comp_ly["Difference"]):
        ax.text(i, v + 0.1 * np.sign(v), f"{v:+.1f}", ha="center", fontsize=9)
    plt.tight_layout()
    plt.savefig(FIGURES / "climate_forecast_vs_lastyear.png", dpi=150)
    plt.show()

    comp_ly.to_csv(METRICS / "forecast_vs_lastyear.csv")
else:
    print("⚠ Temperature data not available for comparison")

## §3 — Is the Forecast Hotter Than the Decade Average?

We compare against the 10-year (2015–2024) historical average for the same months. A positive anomaly suggests a warming signal.

In [ ]:
# ─── §3: Forecast vs Decade Average ──────────────────────────────────────

fc_months = list(range(fc_month_start, fc_month_end + 1))
if fc_month_start > fc_month_end:
    fc_months = list(range(fc_month_start, 13)) + list(range(1, fc_month_end + 1))

decade = hist[(hist["Year"] >= DECADE_START) & (hist["Year"] <= DECADE_END)
              & (hist["Month"].isin(fc_months))]

if temp_col in decade.columns and not fc_temp.empty:
    decade_avg = decade.groupby("City")[temp_col].mean()

    comp_dec = pd.DataFrame({
        "Forecast_Temp": fc_temp,
        "Decade_Avg": decade_avg,
    }).dropna()
    comp_dec["Anomaly"] = comp_dec["Forecast_Temp"] - comp_dec["Decade_Avg"]
    comp_dec["Above_Normal"] = comp_dec["Anomaly"] > 0
    comp_dec = comp_dec.sort_values("Anomaly", ascending=False)

    print("Temperature: Forecast vs Decade Average (2015-2024, same months)")
    print("=" * 65)
    for city, row in comp_dec.iterrows():
        flag = "↑ ABOVE" if row["Above_Normal"] else "↓ below"
        print(f"  {city:15s}  Forecast={row['Forecast_Temp']:5.1f}°C  "
              f"DecAvg={row['Decade_Avg']:5.1f}°C  "
              f"Anomaly={row['Anomaly']:+.1f}°C  {flag}")

    n_above = comp_dec["Above_Normal"].sum()
    print(f"\n  {n_above}/{len(comp_dec)} cities are ABOVE the decade average")
    print(f"  Average anomaly: {comp_dec['Anomaly'].mean():+.2f}°C")

    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    x = range(len(comp_dec))
    colors = ["#e74c3c" if a > 0 else "#3498db" for a in comp_dec["Anomaly"]]
    ax.bar(x, comp_dec["Anomaly"], color=colors, alpha=0.8, edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels(comp_dec.index, rotation=45, ha="right")
    ax.set_ylabel("Temperature Anomaly (°C)")
    ax.set_title(f"Forecast vs Decade Average ({DECADE_START}-{DECADE_END}) — Temperature Anomaly",
                 fontsize=14)
    ax.axhline(0, color="black", linewidth=0.8)
    for i, v in enumerate(comp_dec["Anomaly"]):
        ax.text(i, v + 0.08 * np.sign(v), f"{v:+.1f}", ha="center", fontsize=9)
    plt.tight_layout()
    plt.savefig(FIGURES / "climate_forecast_vs_decade.png", dpi=150)
    plt.show()

    comp_dec.to_csv(METRICS / "forecast_vs_decade.csv")
else:
    print("⚠ Decade comparison not available")

## §4 — Climate Shift Signal

We look at long-term trends:
- **Annual mean temperature** — is there a warming trend since 2012?
- **Extreme heat days** (temperature > 35°C) per year
- **Dry-day trends** — is precipitation declining?

**Important:** These signals suggest possible trends but do not constitute proof of climate change for Azerbaijan specifically.

In [ ]:
# ─── §4: Climate shift signal ─────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Long-Term Climate Signals — Azerbaijan (2012–Present)", fontsize=16)

# 4a. Annual mean temperature trend
if temp_col in hist.columns:
    yearly_temp = hist.groupby("Year")[temp_col].mean()
    ax = axes[0, 0]
    ax.plot(yearly_temp.index, yearly_temp.values, "o-", color="#e74c3c", lw=2)
    # Linear trend
    z = np.polyfit(yearly_temp.index, yearly_temp.values, 1)
    p = np.poly1d(z)
    ax.plot(yearly_temp.index, p(yearly_temp.index), "--", color="gray", lw=1.5,
            label=f"Trend: {z[0]:+.3f}°C/year")
    ax.set_xlabel("Year")
    ax.set_ylabel("Mean Temperature (°C)")
    ax.set_title("Annual Mean Temperature Trend")
    ax.legend()
    print(f"Temperature trend: {z[0]:+.4f}°C per year")

# 4b. Extreme heat days per year (Tmax > 35°C)
tmax_col = "Temperature_C_max"
if tmax_col in hist.columns:
    hist["extreme_heat"] = (hist[tmax_col] > 35).astype(int)
    yearly_extreme = hist.groupby("Year")["extreme_heat"].sum()
    ax = axes[0, 1]
    ax.bar(yearly_extreme.index, yearly_extreme.values, color="#e67e22", alpha=0.8)
    z2 = np.polyfit(yearly_extreme.index, yearly_extreme.values, 1)
    ax.plot(yearly_extreme.index, np.poly1d(z2)(yearly_extreme.index), "--",
            color="red", lw=2, label=f"Trend: {z2[0]:+.1f} days/year")
    ax.set_xlabel("Year")
    ax.set_ylabel("Extreme Heat Days (Tmax > 35°C)")
    ax.set_title("Extreme Heat Days Per Year")
    ax.legend()
    print(f"Extreme heat trend: {z2[0]:+.2f} days per year")

# 4c. Annual precipitation trend
rain_col = "Rain_mm_sum"
if rain_col in hist.columns:
    yearly_rain = hist.groupby("Year")[rain_col].sum() / hist.groupby("Year")["City"].nunique()
    ax = axes[1, 0]
    ax.bar(yearly_rain.index, yearly_rain.values, color="#3498db", alpha=0.8)
    z3 = np.polyfit(yearly_rain.index, yearly_rain.values, 1)
    ax.plot(yearly_rain.index, np.poly1d(z3)(yearly_rain.index), "--",
            color="navy", lw=2, label=f"Trend: {z3[0]:+.1f} mm/year")
    ax.set_xlabel("Year")
    ax.set_ylabel("Annual Precipitation (mm, avg per city)")
    ax.set_title("Annual Precipitation Trend")
    ax.legend()
    print(f"Precipitation trend: {z3[0]:+.2f} mm per year")

# 4d. Annual fire count trend
if "Fire_Occurred" in hist.columns:
    yearly_fires = hist.groupby("Year")["Fire_Occurred"].sum()
    ax = axes[1, 1]
    ax.bar(yearly_fires.index, yearly_fires.values, color="#e74c3c", alpha=0.8)
    z4 = np.polyfit(yearly_fires.index, yearly_fires.values, 1)
    ax.plot(yearly_fires.index, np.poly1d(z4)(yearly_fires.index), "--",
            color="darkred", lw=2, label=f"Trend: {z4[0]:+.1f} fires/year")
    ax.set_xlabel("Year")
    ax.set_ylabel("Fire Events")
    ax.set_title("Annual Fire Count Trend")
    ax.legend()
    print(f"Fire count trend: {z4[0]:+.2f} fires per year")

plt.tight_layout()
plt.savefig(FIGURES / "climate_long_term_trends.png", dpi=150)
plt.show()

## §5 — Will Wildfire Risk Increase?

Compare predicted wildfire risk for the next 30 days against:
- Last year's same period (actual fire rate)
- Historical same-month average fire rate

In [ ]:
# ─── §5: Wildfire risk comparison ─────────────────────────────────────────

# Forecast mean fire probability per city
fc_risk = risk.groupby("City")["fire_probability"].mean()

# Last year actual fire rate
ly_fire = hist[(hist["Date"] >= ly_start) & (hist["Date"] <= ly_end)]
if "Fire_Occurred" in ly_fire.columns:
    ly_fire_rate = ly_fire.groupby("City")["Fire_Occurred"].mean()
else:
    ly_fire_rate = pd.Series(dtype=float)

# Decade historical fire rate (same months)
dec_fire = hist[(hist["Year"] >= DECADE_START) & (hist["Year"] <= DECADE_END)
                & (hist["Month"].isin(fc_months))]
if "Fire_Occurred" in dec_fire.columns:
    dec_fire_rate = dec_fire.groupby("City")["Fire_Occurred"].mean()
else:
    dec_fire_rate = pd.Series(dtype=float)

# Build comparison table
risk_comp = pd.DataFrame({
    "Forecast_Risk": fc_risk,
    "LastYear_FireRate": ly_fire_rate,
    "Decade_FireRate": dec_fire_rate,
}).dropna()

risk_comp["vs_LastYear"] = risk_comp["Forecast_Risk"] - risk_comp["LastYear_FireRate"]
risk_comp["vs_Decade"]   = risk_comp["Forecast_Risk"] - risk_comp["Decade_FireRate"]
risk_comp["Risk_Increasing"] = (risk_comp["vs_LastYear"] > 0) | (risk_comp["vs_Decade"] > 0)
risk_comp = risk_comp.sort_values("Forecast_Risk", ascending=False)

print("Wildfire Risk Comparison")
print("=" * 80)
for city, row in risk_comp.iterrows():
    ly_arrow = "↑" if row["vs_LastYear"] > 0 else "↓"
    dc_arrow = "↑" if row["vs_Decade"] > 0 else "↓"
    print(f"  {city:15s}  Risk={row['Forecast_Risk']*100:5.1f}%  "
          f"vsLY={row['vs_LastYear']*100:+5.1f}%{ly_arrow}  "
          f"vsDec={row['vs_Decade']*100:+5.1f}%{dc_arrow}  "
          f"{'⚠ INCREASING' if row['Risk_Increasing'] else ''}")

n_increasing = risk_comp["Risk_Increasing"].sum()
print(f"\n  {n_increasing}/{len(risk_comp)} cities show INCREASING risk vs last year or decade")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# vs Last Year
ax = axes[0]
x = range(len(risk_comp))
ax.bar(x, risk_comp["vs_LastYear"] * 100,
       color=["#e74c3c" if v > 0 else "#2ecc71" for v in risk_comp["vs_LastYear"]],
       alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(risk_comp.index, rotation=45, ha="right")
ax.set_ylabel("Risk Change (%)")
ax.set_title("Forecast Risk vs Last Year", fontsize=13)
ax.axhline(0, color="black", linewidth=0.8)

# vs Decade
ax = axes[1]
ax.bar(x, risk_comp["vs_Decade"] * 100,
       color=["#e74c3c" if v > 0 else "#2ecc71" for v in risk_comp["vs_Decade"]],
       alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(risk_comp.index, rotation=45, ha="right")
ax.set_ylabel("Risk Change (%)")
ax.set_title(f"Forecast Risk vs Decade Average ({DECADE_START}-{DECADE_END})", fontsize=13)
ax.axhline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.savefig(FIGURES / "climate_risk_comparison.png", dpi=150)
plt.show()

risk_comp.to_csv(METRICS / "risk_comparison.csv")

## §6 — Jury-Friendly Final Conclusions

A concise, non-technical summary of findings suitable for a live presentation.

In [ ]:
# ─── §6: Jury-friendly final conclusions ──────────────────────────────────

print("=" * 70)
print("   AZERBAIJAN WILDFIRE & CLIMATE INTELLIGENCE — KEY FINDINGS")
print("=" * 70)

# Highest risk cities
print("\n🔥 HIGHEST WILDFIRE RISK CITIES (next 30 days):")
top_risk = risk_comp.head(5)
for i, (city, row) in enumerate(top_risk.iterrows(), 1):
    print(f"   {i}. {city} — {row['Forecast_Risk']*100:.1f}% fire probability")

# Hottest cities
if "Forecast_Temp" in comp_ly.columns if "comp_ly" in dir() else False:
    print("\n🌡 HOTTEST CITIES (forecast period):")
    hottest = comp_ly.nlargest(5, "Forecast_Temp")
    for i, (city, row) in enumerate(hottest.iterrows(), 1):
        print(f"   {i}. {city} — {row['Forecast_Temp']:.1f}°C "
              f"({row['Difference']:+.1f}°C vs last year)")

# Biggest risk increase
print("\n📈 BIGGEST RISK INCREASE vs Last Year:")
if "vs_LastYear" in risk_comp.columns:
    biggest_inc = risk_comp.nlargest(5, "vs_LastYear")
    for i, (city, row) in enumerate(biggest_inc.iterrows(), 1):
        print(f"   {i}. {city} — {row['vs_LastYear']*100:+.1f}% change")

# Climate signals
print("\n📊 CLIMATE SHIFT SIGNALS:")
if 'z' in dir() and z[0] > 0:
    print(f"   • Temperature trend: {z[0]:+.3f}°C per year (warming indication)")
else:
    print(f"   • Temperature trend: insufficient data")

if 'z2' in dir() and z2[0] > 0:
    print(f"   • Extreme heat days: {z2[0]:+.1f} more days per year (increasing)")
else:
    print(f"   • Extreme heat days: no clear increase")

if 'z3' in dir() and z3[0] < 0:
    print(f"   • Precipitation: {z3[0]:+.1f} mm per year (declining — drier conditions)")
else:
    print(f"   • Precipitation: no clear decline")

# Main drivers
print("\n🔑 MAIN DRIVERS OF FIRE RISK:")
print("   1. High temperature + low humidity combination")
print("   2. Extended dry periods (consecutive days without rain)")
print("   3. High wind speed during dry conditions")
print("   4. Seasonal pattern — peak risk May through September")
print("   5. Vegetation density — forested areas with drought stress")

print("\n⚠ DISCLAIMER:")
print("   These findings represent statistical signals and model predictions.")
print("   They should be interpreted as risk indicators, not absolute forecasts.")
print("   Local conditions, human factors, and sudden weather changes")
print("   can significantly affect actual fire occurrence.")

# Save summary
summary_data = []
for city, row in risk_comp.iterrows():
    summary_data.append({
        "City": city,
        "Forecast_Risk_Pct": round(row["Forecast_Risk"] * 100, 1),
        "vs_LastYear_Pct": round(row["vs_LastYear"] * 100, 1),
        "vs_Decade_Pct": round(row["vs_Decade"] * 100, 1),
        "Risk_Increasing": row["Risk_Increasing"],
    })
pd.DataFrame(summary_data).to_csv(METRICS / "climate_summary.csv", index=False)

print(f"\n{'='*70}")
print(f"  All analysis outputs saved to: {METRICS}")
print(f"  All figures saved to: {FIGURES}")
print(f"{'='*70}")